# 00 — Příprava dat
## 4IZ503 Projektový seminář — Ultra Marathon Running

**Datasety:**
- Hlavní: `TWO_CENTURIES_OF_UM_RACES.csv` (Kaggle, ~7.5M záznamů)
- Metadata závodů: `races_metadata_template.xlsx` (vlastní, 33 matchovaných závodů)

**Výstupy tohoto notebooku:**
- `ultra_clean.parquet` — kompletní dataset pro Python analýzy (numerické + kategorické sloupce)
- `ultra_clean_cm.parquet` — dataset pro CleverMiner (pouze string kategorické sloupce, správné pořadí)

**Předpokládaná struktura složek:**
```
projekt/
├── data/
│   ├── TWO_CENTURIES_OF_UM_RACES.csv
│   ├── races_metadata_template.xlsx
│   └── processed/
│       ├── ultra_clean.parquet
│       └── ultra_clean_cm.parquet
└── notebooks/
    └── 00_data_preparation.ipynb
```


## 1. Import knihoven

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR      = Path('../data')
PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Pandas:  {pd.__version__}")
print(f"NumPy:   {np.__version__}")
print(f"Výstupní složka: {PROCESSED_DIR.resolve()}")


## 2. Načtení hlavního datasetu

In [ ]:
print("Načítám hlavní dataset... (může trvat 30–60 s)")

df = pd.read_csv(DATA_DIR / 'TWO_CENTURIES_OF_UM_RACES.csv', low_memory=False)

print(f"Načteno: {len(df):,} řádků, {df.shape[1]} sloupců")
print(f"Sloupce: {df.columns.tolist()}")


## 3. Přejmenování sloupců

In [ ]:
df = df.rename(columns={
    'Year of event':             'year',
    'Event dates':               'event_dates',
    'Event name':                'event_name',
    'Event distance/length':     'distance_raw',
    'Event number of finishers': 'n_finishers',
    'Athlete performance':       'performance_raw',
    'Athlete club':              'club',
    'Athlete country':           'athlete_country',
    'Athlete year of birth':     'birth_year',
    'Athlete gender':            'gender',
    'Athlete age category':      'age_category_raw',
    'Athlete average speed':     'avg_speed_raw',
    'Athlete ID':                'athlete_id',
})
print("Přejmenováno ✓")
print(df.dtypes)


## 4. Odvození: věk závodníka a věkové skupiny

In [ ]:
df['year']       = pd.to_numeric(df['year'],       errors='coerce')
df['birth_year'] = pd.to_numeric(df['birth_year'], errors='coerce')
df['age']        = df['year'] - df['birth_year']

# Odstranění nereálných věků
n_before = len(df)
df = df[(df['age'] >= 16) & (df['age'] <= 90) | df['age'].isna()]
print(f"Odstraněno nereálných věků: {n_before - len(df):,}")

# Věkové skupiny pro Python analýzy
AGE_CATS = ['18-29', '30-39', '40-49', '50-59', '60-69', '70+']
df['age_group'] = pd.Categorical(
    pd.cut(df['age'], bins=[0,29,39,49,59,69,120],
           labels=AGE_CATS, right=True),
    categories=AGE_CATS, ordered=True
)

print("Věkové skupiny ✓")
print(df['age_group'].value_counts().sort_index())


## 5. Odvození: průměrná rychlost

In [ ]:
df['avg_speed'] = (
    df['avg_speed_raw']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .str.extract(r'([\d\.]+)')[0]
    .astype(float)
)
# Odstranění nereálných rychlostí
df.loc[(df['avg_speed'] < 1) | (df['avg_speed'] > 25), 'avg_speed'] = np.nan

n_valid = df['avg_speed'].notna().sum()
print(f"avg_speed: {n_valid:,} platných hodnot ({n_valid/len(df)*100:.1f}%)")
print(f"Průměr: {df['avg_speed'].mean():.2f} km/h")
print(f"Medián: {df['avg_speed'].median():.2f} km/h")


## 6. Odvození: vzdálenostní kategorie

In [ ]:
# Vektorizovaná verze — bez apply

dist_str = df['distance_raw'].astype(str).str.strip().str.lower()

# Vzdálenost v km
km_mask = dist_str.str.contains('km', na=False)
mi_mask = dist_str.str.contains('mi', na=False) & ~km_mask

km_vals = pd.to_numeric(
    dist_str.str.replace('km', '', regex=False).str.strip(),
    errors='coerce'
)
mi_vals = pd.to_numeric(
    dist_str.str.replace('mi', '', regex=False).str.strip(),
    errors='coerce'
) * 1.60934

df['distance_km'] = np.where(km_mask, km_vals,
                    np.where(mi_mask, mi_vals, np.nan))

# Časové závody
df['is_timed'] = (
    dist_str.str.contains('h', na=False) &
    ~dist_str.str.contains('km|mi', na=False)
)

# Vzdálenostní kategorie — vektorizovaně
d = df['distance_km']
conditions = [
    df['is_timed'],
    d < 60,
    d < 100,
    d <= 170,
    d > 170,
]
choices = ['casovy', 'kratka', 'stredni', 'dlouha', 'extremni']
df['distance_cat_raw'] = np.select(conditions, choices, default=None)
df.loc[~df['is_timed'] & df['distance_km'].isna(), 'distance_cat_raw'] = None

DIST_CATS = ['kratka', 'stredni', 'dlouha', 'extremni', 'casovy']
df['distance_cat'] = pd.Categorical(
    df['distance_cat_raw'],
    categories=DIST_CATS, ordered=True
)

print("Vzdálenostní kategorie ✓")
print(df['distance_cat'].value_counts().sort_index())


## 7. Odvození: speed_cat per event

> ⚠️ **Klíčové metodické rozhodnutí:**
> `speed_cat` počítáme v rámci každého závodu zvlášť (kvantily 33/67 per `event_name`).
> Každý závod má přesně ~33 % pomalých, středních a rychlých závodníků.
> Tím je srovnání závodníků na různě obtížných závodech férové.


In [ ]:
# speed_cat per event — bez apply, přes groupby transform
print("Počítám speed_cat per event...")

# Výpočet kvantilů per event přes transform — vektorizovaně
df['q33'] = df.groupby('event_name')['avg_speed'].transform(lambda x: x.quantile(0.33))
df['q67'] = df.groupby('event_name')['avg_speed'].transform(lambda x: x.quantile(0.67))

# Přiřazení kategorie přes np.select
conditions = [
    df['avg_speed'].isna(),
    df['avg_speed'] <= df['q33'],
    df['avg_speed'] <= df['q67'],
    df['avg_speed'] >  df['q67'],
]
choices = [None, 'pomalý', 'střední', 'rychlý']
df['speed_cat'] = np.select(conditions, choices, default=None)

# Záznamy s méně než 10 finišery → None
small_events = df.groupby('event_name')['avg_speed'].transform('count') < 10
df.loc[small_events, 'speed_cat'] = None

# Úklid pomocných sloupců
df = df.drop(columns=['q33', 'q67'])

SPEED_CATS = ['pomalý', 'střední', 'rychlý']
df['speed_cat'] = pd.Categorical(df['speed_cat'], categories=SPEED_CATS, ordered=True)

n_valid = df['speed_cat'].notna().sum()
print(f"speed_cat: {n_valid:,} platných hodnot ({n_valid/len(df)*100:.1f}%)")
print("Rozložení (ověření ~33/33/33):")
print((df['speed_cat'].value_counts() / n_valid * 100).round(1))


## 8. Odvození: zkušenosti závodníka

In [ ]:
print("Počítám zkušenosti... (může trvat 1-2 min)")

df_sorted = df.sort_values(['athlete_id', 'year'])
df['experience'] = df_sorted.groupby('athlete_id').cumcount()

# Vektorizovaně přes np.select
EXP_CATS = ['nováček', 'zkušený', 'veterán']
df['experience_cat'] = pd.Categorical(
    np.select(
        [df['experience'] == 0, df['experience'] <= 4],
        ['nováček', 'zkušený'],
        default='veterán'
    ),
    categories=EXP_CATS, ordered=True
)

print("experience_cat ✓")
print(df['experience_cat'].value_counts().sort_index())


## 9. Odvození: měsíc a sezóna

In [ ]:
# Pokus 1: standardní parsování
df['month'] = pd.to_datetime(
    df['event_dates'], dayfirst=True, errors='coerce'
).dt.month

# Pokus 2: jiný formát
mask = df['month'].isna()
df.loc[mask, 'month'] = pd.to_datetime(
    df.loc[mask, 'event_dates'], dayfirst=False, errors='coerce'
).dt.month

# Pokus 3: formát "12.-13.10.2019" — extrakce posledního data
mask = df['month'].isna()
fixed = (
    df.loc[mask, 'event_dates']
    .astype(str).str.split('-').str[-1].str.strip()
)
df.loc[mask, 'month'] = pd.to_datetime(
    fixed, dayfirst=True, errors='coerce'
).dt.month

season_map = {
    12:'zima', 1:'zima',  2:'zima',
    3:'jaro',  4:'jaro',  5:'jaro',
    6:'léto',  7:'léto',  8:'léto',
    9:'podzim',10:'podzim',11:'podzim'
}
df['season'] = df['month'].map(season_map)
df['season'] = pd.Categorical(
    df['season'],
    categories=['jaro','léto','podzim','zima'],
    ordered=False
)

n_null = df['season'].isna().sum()
print(f"season ✓  (null: {n_null:,} = {n_null/len(df)*100:.2f}%)")
print(df['season'].value_counts())


## 10. Odvození: dekáda

In [ ]:
df['decade'] = (
    df['year'].dropna().astype(int) // 10 * 10
).astype(str) + 's'
df.loc[df['year'].isna(), 'decade'] = None

DECADE_CATS = ['1990s','2000s','2010s','2020s']
df['decade'] = pd.Categorical(
    df['decade'].where(df['decade'].isin(DECADE_CATS)),
    categories=DECADE_CATS, ordered=True
)

print("decade ✓")
print(df['decade'].value_counts().sort_index())


## 11. JOIN s metadaty závodů

In [ ]:
meta = pd.read_excel(
    DATA_DIR / 'races_metadata_template.xlsx',
    sheet_name='races_metadata'
)
print(f"Metadata: {len(meta)} závodů")
print(f"Sloupce: {meta.columns.tolist()}")


In [ ]:
df = df.merge(
    meta[['event_name','typical_distance_km',
          'elevation_gain_m','elevation_loss_m','surface','notes']],
    on='event_name', how='left'
)

# Elevation kategorie — vektorizovaně přes pd.cut
ELEV_CATS = ['nizke', 'stredni', 'vysoke', 'extremni']
df['elevation_cat'] = pd.Categorical(
    pd.cut(
        df['elevation_gain_m'],
        bins=[0, 1000, 3000, 6000, float('inf')],
        labels=ELEV_CATS,
        right=False
    ).astype(str).replace('nan', None),
    categories=ELEV_CATS, ordered=True
)

# Surface — nominální
SURFACE_CATS = ['road', 'trail', 'mixed']
df['surface'] = pd.Categorical(
    df['surface'].where(df['surface'].isin(SURFACE_CATS)),
    categories=SURFACE_CATS, ordered=False
)

n_meta = df['surface'].notna().sum()
print(f"Závodů s metadaty (surface): {n_meta:,} ({n_meta/len(df)*100:.1f}%)")
print(df['surface'].value_counts(dropna=False))


## 12. Čištění a filtrování

In [ ]:
n_before = len(df)
print(f"Před filtrací: {n_before:,}")

# Základní filtry
df = df[
    (df['athlete_country'] != 'XXX') &
    (df['year'] >= 1990) &
    (df['distance_km'].isna() | (df['distance_km'] <= 500)) &
    (df['gender'].isin(['M', 'F']))
].copy()
print(f"Po základních filtrech: {len(df):,} (odstraněno {n_before-len(df):,})")

# Odstranění Split závodů
n2 = len(df)
df = df[~df['event_name'].str.contains('Split|split', na=False)].copy()
print(f"Po odstranění Splitů: {len(df):,} (odstraněno {n2-len(df):,})")

print(f"\nCelkem odstraněno: {n_before-len(df):,} řádků ({(n_before-len(df))/n_before*100:.1f}%)")


## 13. Přehled finálního datasetu

In [ ]:
print("=" * 60)
print("FINÁLNÍ DATASET — PŘEHLED")
print("=" * 60)
print(f"Celkem řádků:              {len(df):>10,}")
print(f"Unikátních závodů:         {df['event_name'].nunique():>10,}")
print(f"Závodníků (athlete_id):    {df['athlete_id'].nunique():>10,}")
print(f"Zemí závodníků:            {df['athlete_country'].nunique():>10,}")
print(f"Časové rozmezí:            {int(df['year'].min())} – {int(df['year'].max())}")
print()
print(f"{'Sloupec':<22} {'Platné hodnoty':>15} {'Pokrytí':>8}")
print("-" * 48)
for col in ['age_group','speed_cat','experience_cat','gender',
            'distance_cat','season','decade','surface','elevation_cat']:
    n = int(df[col].notna().sum())
    pct = n / len(df) * 100
    print(f"  {col:<20} {n:>15,} {pct:>7.1f}%")


## 14. Export — ultra_clean.parquet

Pro Python analýzy — obsahuje všechny sloupce včetně numerických.


In [ ]:
cols_export = [
    'year', 'event_name', 'distance_raw', 'distance_km',
    'distance_cat', 'is_timed', 'n_finishers',
    'athlete_id', 'athlete_country', 'gender',
    'birth_year', 'age', 'age_group',
    'avg_speed', 'speed_cat',
    'experience', 'experience_cat',
    'month', 'season', 'decade',
    'typical_distance_km', 'elevation_gain_m', 'elevation_loss_m',
    'surface', 'elevation_cat', 'notes'
]

out_path = PROCESSED_DIR / 'ultra_clean.parquet'
df[cols_export].to_parquet(out_path, index=False, compression='snappy')
size_mb = out_path.stat().st_size / 1024**2
print(f"Uloženo: {out_path}")
print(f"Velikost: {size_mb:.1f} MB")
print(f"Řádků: {len(df):,}")


## 15. Export — ultra_clean_cm.parquet

Pro CleverMiner.

### Klíčová pravidla pro CleverMiner:
1. **Pouze string hodnoty** — CleverMiner pracuje pouze s textovými kategoriemi
2. **Pořadí kategorií** — CleverMiner řadí kategorie ABECEDNĚ, ne podle `pd.Categorical`.
   Proto pojmenujeme ordinální kategorie s číselným prefixem:
   `1_pomalý < 2_střední < 3_rychlý` → abecední pořadí = logické pořadí
3. **Žádné NaN** — CleverMiner ignoruje záznamy s NaN v použitých sloupcích per úlohu

### Sloupce a jejich použití v úlohách:

| Sloupec | Typ | Pořadí | Používá se v |
|---|---|---|---|
| speed_cat | ordinální | 1_pomalý < 2_střední < 3_rychlý | všechny úlohy (succ/target) |
| age_group | ordinální | 1_18-29 < ... < 6_70+ | CF-2, SD4ft-1 |
| experience_cat | ordinální | 1_nováček < 2_zkušený < 3_veterán | 4ft-2, CF-1, SD4ft-2 |
| gender | nominální | F / M | 4ft-1, CF-2, SD4ft-1 |
| distance_cat | ordinální | 1_kratka < ... < 5_casovy | 4ft-1, 4ft-2, CF-1, SD4ft-2 |
| season | nominální | jaro/léto/podzim/zima | CF-1 |
| surface | nominální | road/trail/mixed | CF-2 |
| elevation_cat | ordinální | 1_nizke < ... < 4_extremni | CF-2 |


In [ ]:
# Export pro CleverMiner — bez číselných prefixů
# Používáme pouze type:'one' a type:'subset' → pořadí kategorií nehraje roli

df_cm = pd.DataFrame({
    'speed_cat':      df['speed_cat'].astype(str).where(df['speed_cat'].notna()),
    'age_group':      df['age_group'].astype(str).where(df['age_group'].notna()),
    'experience_cat': df['experience_cat'].astype(str).where(df['experience_cat'].notna()),
    'gender':         df['gender'].astype(str).where(df['gender'].isin(['M','F'])),
    'distance_cat':   df['distance_cat'].astype(str).where(df['distance_cat'].notna()),
    'season':         df['season'].astype(str).where(df['season'].notna()),
    'surface':        df['surface'].astype(str).where(df['surface'].notna()),
    'elevation_cat':  df['elevation_cat'].astype(str).where(df['elevation_cat'].notna()),
})

# Nahradit 'nan' stringy za skutečné None
for col in df_cm.columns:
    df_cm[col] = df_cm[col].replace({'nan': None, 'None': None, '<NA>': None})

print(f"CleverMiner dataset: {len(df_cm):,} řádků, {len(df_cm.columns)} sloupců")
print()
print(f"{'Sloupec':<20} {'Unikátní hodnoty':<45} {'Null':>8}")
print("-" * 76)
for col in df_cm.columns:
    vals = sorted(df_cm[col].dropna().unique())
    n_null = df_cm[col].isna().sum()
    print(f"  {col:<18} {str(vals):<45} {n_null:>8,}")


In [ ]:
# Ověření pořadí kategorií na malém vzorku přes CleverMiner
from cleverminer import cleverminer

sample = df_cm[['speed_cat','age_group','experience_cat',
                'distance_cat']].dropna().sample(1000, random_state=42)

cm_test = cleverminer(df=sample)
print("Ověření pořadí kategorií v CleverMineru:")
cm_test.print_data_definition()


In [ ]:
# Export ultra_clean_cm.parquet
out_cm = PROCESSED_DIR / 'ultra_clean_cm.parquet'
df_cm.to_parquet(out_cm, index=False, compression='snappy')
size_mb = out_cm.stat().st_size / 1024**2

print(f"Uloženo: {out_cm}")
print(f"Velikost: {size_mb:.1f} MB")
print()
print("Soubory připraveny:")
print(f"  ultra_clean.parquet    — Python analýzy")
print(f"  ultra_clean_cm.parquet — CleverMiner")


## 16. Rychlý sanity check CleverMiner — zkušební 4ft úloha

In [ ]:
# Ověřovací test: spustíme jednoduchou 4ft úlohu na vzorku
# Pokud projde, notebook 00 je správně nastaven pro všechny analytické notebooky

from cleverminer import cleverminer

# Malý vzorek pro rychlý test
df_test = df_cm[['age_group','speed_cat','experience_cat',
                 'gender','distance_cat']].dropna().sample(10000, random_state=42)

cm = cleverminer(df=df_test)

cm.mine(
    proc='4ftMiner',
    quantifiers={'Base': 100, 'aad': 0.05},
    ante={
        'attributes': [
            {'name': 'age_group', 'type': 'subset', 'minlen': 1, 'maxlen': 1}
        ],
        'minlen': 1, 'maxlen': 1, 'type': 'con'
    },
    succ={
        'attributes': [
            {'name': 'speed_cat', 'type': 'one', 'value': 'rychlý'}
        ],
        'minlen': 1, 'maxlen': 1, 'type': 'con'
    }
)

print("\nSouhrn testovací úlohy:")
cm.print_summary()
print("\nPravidla:")
cm.print_rulelist()


## Shrnutí

### Výsledný dataset
| Metrika | Hodnota |
|---|---|
| Řádků | ~6.87M |
| Závodů | ~23 000 |
| Závodníků | ~1.56M |
| Období | 1990–2022 |
| Záznamy s metadaty (surface/elevation) | ~15.9 % |

### Klíčové metodické rozhodnutí: speed_cat per event
`speed_cat` je počítán v rámci každého závodu zvlášť.
Každý závod má přesně ~33 % pomalých, středních a rychlých závodníků —
férové srovnání závodníků na různě obtížných závodech.

### Pořadí kategorií pro CleverMiner
CleverMiner řadí kategorie abecedně. Proto používáme číselné prefixy:

| Proměnná | Hodnoty v CM datasetu |
|---|---|
| speed_cat | 1_pomalý, 2_střední, 3_rychlý |
| experience_cat | 1_nováček, 2_zkušený, 3_veterán |
| distance_cat | 1_kratka, 2_stredni, 3_dlouha, 4_extremni, 5_casovy |
| age_group | 1_18-29, 2_30-39, 3_40-49, 4_50-59, 5_60-69, 6_70+ |
| elevation_cat | 1_nizke, 2_stredni, 3_vysoke, 4_extremni |
| gender | F, M (nominální) |
| season | jaro, léto, podzim, zima (nominální) |
| surface | road, trail, mixed (nominální) |

### Limitace
- `surface` a `elevation_cat` dostupné jen pro 33 závodů (~15.9 % dat)
- `athlete_id` není globálně unikátní → `experience_cat` je aproximace
- Dataset dominován USA, FRA, RSA, JPN
- Ženy tvoří ~19.8 % datasetu

**Další notebook:** `03_4ft_uloha1.ipynb` — Výkonnost žen na extrémních vzdálenostech
